In [ ]:
# 1. Instalar Conda-Colab para gestionar paquetes científicos
!pip install -q condacolab
import condacolab
condacolab.install()


📢 Announcement 📢
condacolab==0.2 will be released soon! Try it with:

    !pip install -q https://github.com/conda-incubator/condacolab/archive/main.zip
    import condacolab
    condacolab.install()

0.2.x introduces a new installation method based on Pixi, with customizable Python versions.
This may be breaking for your workflow. If that's the case, please report it at
https://github.com/conda-incubator/condacolab and pin your `pip install` command to
condacolab==0.1 as a workaround.

⏬ Downloading https://github.com/conda-forge/miniforge/releases/download/26.3.2-3/Miniforge3-26.3.2-3-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:08
🔁 Restarting kernel...


In [ ]:
!mamba install -y -c conda-forge -c bioconda --no-pin gromacs=*=*cuda* cuda-version=12.6
!ln -sf /usr/local/bin/gmx_mpi /usr/bin/gmx
!ln -sf /usr/local/bin/gmx_mpi /usr/local/bin/gmx

conda-forge/linux-64                                        Using cache
conda-forge/noarch                                          Using cache
bioconda/linux-64                                           Using cache
bioconda/noarch                                             Using cache

Pinned packages:

  - python=3.13


Transaction

  Prefix: /usr/local

  All requested packages already installed


Transaction starting

Transaction finished



In [ ]:
!gmx --version

                 :-) GROMACS - gmx_mpi, 2025.3-conda_forge (-:

Executable:   /usr/local/bin.AVX2_256/gmx_mpi
Data prefix:  /usr/local
Working dir:  /content
Command line:
  gmx_mpi --version

GROMACS version:     2025.3-conda_forge
Precision:           mixed
Memory model:        64 bit
MPI library:         MPI
MPI library version: Open MPI v5.0.8, package: Open MPI conda@e2c916615969 Distribution, ident: 5.0.8, repo rev: v5.0.8, May 30, 2025
OpenMP support:      enabled (GMX_OPENMP_MAX_THREADS = 128)
GPU support:         CUDA
NBNxM GPU setup:     super-cluster 2x2x2 / cluster 8 (cluster-pair splitting on)
SIMD instructions:   AVX2_256
CPU FFT library:     fftw-3.3.11-sse2-avx
GPU FFT library:     cuFFT
Multi-GPU FFT:       none
RDTSCP usage:        disabled
TNG support:         enabled
Hwloc support:       disabled
Tracing support:     disabled
C compiler:          /home/conda/feedstock_root/build_artifacts/gromacs_1756723339265/_build_env/bin/x86_64-conda-linux-gnu-cc GNU 13.4.0
C co

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!mkdir -p "/content/drive/MyDrive/MD_Nanobody_Project/WT"
!mkdir -p "/content/drive/MyDrive/MD_Nanobody_Project/MUT"

import os
BASE_DIR = "/content/drive/MyDrive/MD_Nanobody_Project"
os.chdir(BASE_DIR)
print(f"Directorio actual: {os.getcwd()}")

Directorio actual: /content/drive/MyDrive/MD_Nanobody_Project


In [ ]:
!cp /content/System_3A.pdb "/content/drive/MyDrive/MD_Nanobody_Project/WT/"
!cp /content/System_3B.pdb "/content/drive/MyDrive/MD_Nanobody_Project/MUT/"

cp: cannot stat '/content/System_3A.pdb': No such file or directory
cp: cannot stat '/content/System_3B.pdb': No such file or directory


In [ ]:
!ls *

ions.mdp  minim.mdp  npt.mdp  nvt.mdp  prod.mdp

MUT:
System_3B.pdb

WT:
 box.gro     npt.edr   nvt.xtc			   solv.gro
 em.edr      npt.gro   posre_Protein_chain_A.itp   solv_ions.gro
 em.gro      npt.log   posre_Protein_chain_B.itp   System_3A.pdb
 em.log      npt.tpr   processed.gro		   topol_Protein_chain_A.itp
 em.tpr      npt.xtc   prod.cpt			   topol_Protein_chain_B.itp
 em.trr      nvt.cpt   prod.edr			   topol.top
 index.ndx   nvt.edr   prod.gro			  '#topol.top.1#'
 ions.tpr    nvt.gro   prod.log			  '#topol.top.2#'
 mdout.mdp   nvt.log   prod.tpr
 npt.cpt     nvt.tpr   prod.xtc


In [ ]:
%%writefile /content/drive/MyDrive/MD_Nanobody_Project/ions.mdp
integrator  = steep
emtol       = 1000.0
emstep      = 0.01
nsteps      = 50000
cutoff-scheme = Verlet
coulombtype = cutoff
rcoulomb    = 1.0
rvdw        = 1.0
pbc         = xyz

Overwriting /content/drive/MyDrive/MD_Nanobody_Project/ions.mdp


In [ ]:
%%writefile /content/drive/MyDrive/MD_Nanobody_Project/minim.mdp
integrator  = steep
emtol       = 1000.0
emstep      = 0.01
nsteps      = 50000
cutoff-scheme = Verlet
coulombtype = PME
rcoulomb    = 1.0
rvdw        = 1.0
pbc         = xyz

Overwriting /content/drive/MyDrive/MD_Nanobody_Project/minim.mdp


In [ ]:
%%writefile /content/drive/MyDrive/MD_Nanobody_Project/nvt.mdp
title                   = NVT equilibration
define                  = -DPOSRES
integrator              = md
dt                      = 0.002     ; 2 fs
nsteps                  = 50000     ; 100 ps
nstxout-compressed      = 5000
cutoff-scheme           = Verlet
nstlist                 = 20
coulombtype             = PME
rcoulomb                = 1.0
rvdw                    = 1.0

; Tratamiento de enlaces (Solución al warning)
constraints             = h-bonds
constraint-algorithm    = LINCS

tcoupl                  = v-rescale
tc-grps                 = Protein Non-Protein
tau_t                   = 0.1     0.1
ref_t                   = 300     300
pcoupl                  = no
pbc                     = xyz
continuation            = no
gen_vel                 = yes
gen_temp                = 300
gen_seed                = -1

Overwriting /content/drive/MyDrive/MD_Nanobody_Project/nvt.mdp


In [ ]:
%%writefile /content/drive/MyDrive/MD_Nanobody_Project/npt.mdp
title                   = NPT equilibration
define                  = -DPOSRES
integrator              = md
dt                      = 0.002     ; 2 fs
nsteps                  = 50000     ; 100 ps
nstxout-compressed      = 5000
cutoff-scheme           = Verlet
nstlist                 = 20
coulombtype             = PME
rcoulomb                = 1.0
rvdw                    = 1.0

; Tratamiento de enlaces
constraints             = h-bonds
constraint-algorithm    = LINCS

; Acoplamiento térmico
tcoupl                  = v-rescale
tc-grps                 = Protein Non-Protein
tau_t                   = 0.1     0.1
ref_t                   = 300     300

; Acoplamiento de presión
pcoupl                  = C-rescale
pcoupltype              = isotropic
tau_p                   = 2.0
ref_p                   = 1.0
compressibility         = 4.5e-5
refcoord_scaling        = com       ; <- Corrección obligatoria

pbc                     = xyz
continuation            = yes
gen_vel                 = no

Overwriting /content/drive/MyDrive/MD_Nanobody_Project/npt.mdp


In [ ]:
%%writefile /content/drive/MyDrive/MD_Nanobody_Project/prod.mdp
title                   = Production MD 1 ns
integrator              = md
dt                      = 0.002     ; 2 fs
nsteps                  = 500000    ; 1 ns
nstxout-compressed      = 5000      ; 100 snapshots para gmx_MMPBSA
cutoff-scheme           = Verlet
nstlist                 = 20
coulombtype             = PME
rcoulomb                = 1.0
rvdw                    = 1.0

; Tratamiento de enlaces
constraints             = h-bonds
constraint-algorithm    = LINCS

tcoupl                  = v-rescale
tc-grps                 = Protein Non-Protein
tau_t                   = 0.1     0.1
ref_t                   = 300     300
pcoupl                  = Parrinello-Rahman
pcoupltype              = isotropic
tau_p                   = 2.0
ref_p                   = 1.0
compressibility         = 4.5e-5
pbc                     = xyz
continuation            = yes
gen_vel                 = no

Overwriting /content/drive/MyDrive/MD_Nanobody_Project/prod.mdp


In [ ]:
%%bash
cd "/content/drive/MyDrive/MD_Nanobody_Project/WT"
BASE="/content/drive/MyDrive/MD_Nanobody_Project"

echo "=== 6. NVT (100 ps) ==="
gmx grompp -f $BASE/nvt.mdp -c em.gro -r em.gro -p topol.top -n index.ndx -o nvt.tpr
gmx mdrun -v -deffnm nvt -nb gpu -pme gpu -bonded gpu

echo "=== 7. NPT (100 ps) ==="
gmx grompp -f $BASE/npt.mdp -c nvt.gro -r nvt.gro -t nvt.cpt -p topol.top -n index.ndx -o npt.tpr
gmx mdrun -v -deffnm npt -nb gpu -pme gpu -bonded gpu

echo "=== 8. Production (1 ns) ==="
gmx grompp -f $BASE/prod.mdp -c npt.gro -t npt.cpt -p topol.top -n index.ndx -o prod.tpr
gmx mdrun -v -deffnm prod -nb gpu -pme gpu -bonded gpu -update gpu

=== 6. NVT (100 ps) ===
Setting the LD random seed to -905969682

Generated 2145 of the 2145 non-bonded parameter combinations

Generated 2145 of the 2145 1-4 parameter combinations

Excluding 3 bonded neighbours molecule type 'Protein_chain_A'

turning H bonds into constraints...

Excluding 3 bonded neighbours molecule type 'Protein_chain_B'

turning H bonds into constraints...

Excluding 2 bonded neighbours molecule type 'SOL'

turning H bonds into constraints...

Excluding 1 bonded neighbours molecule type 'NA'

turning H bonds into constraints...

Excluding 1 bonded neighbours molecule type 'CL'

turning H bonds into constraints...

Setting gen_seed to -38011659

Velocities were taken from a Maxwell distribution at 300 K

The largest distance between excluded atoms is 0.452 nm between atom 2968 and 3219

Determining Verlet buffer for a tolerance of 0.005 kJ/mol/ps at 300 K

Calculated rlist for 1x1 atom pair-list as 1.097 nm, buffer size 0.097 nm

Set rlist, assuming 4x4 atom pair-

                :-) GROMACS - gmx grompp, 2025.3-conda_forge (-:

Executable:   /usr/local/bin.AVX2_256/gmx_mpi
Data prefix:  /usr/local
Working dir:  /content/drive/MyDrive/MD_Nanobody_Project/WT
Command line:
  gmx_mpi grompp -f /content/drive/MyDrive/MD_Nanobody_Project/nvt.mdp -c em.gro -r em.gro -p topol.top -n index.ndx -o nvt.tpr

Ignoring obsolete mdp entry 'title'
Generating 1-4 interactions: fudge = 0.5
Number of degrees of freedom in T-Coupling group Protein is 9547.78
Number of degrees of freedom in T-Coupling group non-Protein is 118536.23

NOTE 1 [file /content/drive/MyDrive/MD_Nanobody_Project/nvt.mdp]:
  Removing center of mass motion in the presence of position restraints
  might cause artifacts. When you are using position restraints to
  equilibrate a macro-molecule, the artifacts are usually negligible.


There was 1 NOTE

Back Off! I just backed up nvt.tpr to ./#nvt.tpr.1#

GROMACS reminds you: "With a Lead Filled Snowshoe" (F. Zappa)

                :-) GROMACS -

In [ ]:
%%bash
cd "/content/drive/MyDrive/MD_Nanobody_Project/WT"
BASE="/content/drive/MyDrive/MD_Nanobody_Project"

echo "=== 7. NPT (100 ps) ==="
gmx grompp -f $BASE/npt.mdp -c nvt.gro -r nvt.gro -t nvt.cpt -p topol.top -n index.ndx -o npt.tpr
gmx mdrun -v -deffnm npt -nb gpu -pme gpu -bonded gpu

echo "=== 8. Production (1 ns) ==="
gmx grompp -f $BASE/prod.mdp -c npt.gro -t npt.cpt -p topol.top -n index.ndx -o prod.tpr
gmx mdrun -v -deffnm prod -nb gpu -pme gpu -bonded gpu -update gpu

In [ ]:
%%bash
cd "/content/drive/MyDrive/MD_Nanobody_Project/MUT"
BASE="/content/drive/MyDrive/MD_Nanobody_Project"

echo "=== 1. pdb2gmx ==="
printf "6\n1\n" | gmx pdb2gmx -f System_3B.pdb -o processed.gro -p topol.top -ignh

echo "=== 2. Editconf & Solvate ==="
gmx editconf -f processed.gro -o box.gro -c -d 1.2 -bt dodecahedron
gmx solvate -cp box.gro -cs spc216.gro -o solv.gro -p topol.top

echo "=== 3. Genion (0.15 M NaCl) ==="
gmx grompp -f $BASE/ions.mdp -c solv.gro -p topol.top -o ions.tpr -maxwarn 1
printf "SOL\n" | gmx genion -s ions.tpr -o solv_ions.gro -p topol.top -pname NA -nname CL -neutral -conc 0.15

echo "=== 4. Energy Minimization ==="
gmx grompp -f $BASE/minim.mdp -c solv_ions.gro -p topol.top -o em.tpr
gmx mdrun -v -deffnm em -nb gpu

echo "=== 5. Index File ==="
printf "q\n" | gmx make_ndx -f em.gro -o index.ndx

echo "=== 6. NVT (100 ps) ==="
gmx grompp -f $BASE/nvt.mdp -c em.gro -r em.gro -p topol.top -n index.ndx -o nvt.tpr
gmx mdrun -v -deffnm nvt -nb gpu -pme gpu -bonded gpu

echo "=== 7. NPT (100 ps) ==="
gmx grompp -f $BASE/npt.mdp -c nvt.gro -r nvt.gro -t nvt.cpt -p topol.top -n index.ndx -o npt.tpr
gmx mdrun -v -deffnm npt -nb gpu -pme gpu -bonded gpu

echo "=== 8. Production (1 ns) ==="
gmx grompp -f $BASE/prod.mdp -c npt.gro -t npt.cpt -p topol.top -n index.ndx -o prod.tpr
gmx mdrun -v -deffnm prod -nb gpu -pme gpu -bonded gpu -update gpu

In [ ]:
%%bash
cd "/content/drive/MyDrive/MD_Nanobody_Project/WT"

# 1. Eliminar saltos a través de las condiciones de contorno periódicas (-pbc nojump)
# Seleccionar grupo "Protein" (1)
printf "1\n" | gmx trjconv -s prod.tpr -f prod.xtc -o prod_nojump.xtc -pbc nojump

# 2. Centrar el complejo y ajustar rototranslaciones (-center, -pbc mol, -ur compact)
# Primer prompt (grupo para centrar): "Protein" (1)
# Segundo prompt (grupo de salida): "Protein" (1)
printf "1\n1\n" | gmx trjconv -s prod.tpr -f prod_nojump.xtc -o prod_clean.xtc -center -pbc mol -ur compact

# 3. Extraer el primer frame como PDB de referencia para gmx_MMPBSA y PyMOL
printf "1\n" | gmx trjconv -s prod.tpr -f prod_clean.xtc -o reference.pdb -dump 0